# 📊 Student Score Prediction — Exploratory Data Analysis


🎯 Objective:
Analyze factors affecting student performance and identify key predictors
using Exploratory Data Analysis (EDA).

We will:
- Understand distribution of exam scores
- Analyze numerical & categorical features
- Identify relationships with target variable
- Generate insights


# ****1. Import Libraries****

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# 2. Load dataset

In [ ]:
df = pd.read_csv("/kaggle/input/competitions/playground-series-s6e1/train.csv")

# 3. Basic Info

In [ ]:
df.info()
df.describe()

# 4. Missing Values and Duplicates

In [ ]:
print("Missing Values:\n", df.isnull().sum())
print("\nDuplicate Rows:", df.duplicated().sum())

# 5. Target Variable Distribution

In [ ]:
TARGET_COL = "exam_score"  # adjust if needed

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df[TARGET_COL], bins=40, kde=True, ax=axes[0])
axes[0].set_title("Distribution of Exam Scores")

axes[1].boxplot(df[TARGET_COL])
axes[1].set_title("Exam Score Boxplot")

plt.show()

# 6. Numerical Feature Correlation

In [ ]:
num_cols = df.select_dtypes(exclude="object").columns

corr = df[num_cols].corr()[TARGET_COL].drop(TARGET_COL).sort_values()

plt.figure(figsize=(8,5))
corr.plot(kind="barh")
plt.title("Correlation with Exam Score")
plt.axvline(0)
plt.show()

# 7. Correlation Heatmap

In [ ]:
plt.figure(figsize=(10,8))

mask = np.triu(np.ones_like(df[num_cols].corr(), dtype=bool))

sns.heatmap(df[num_cols].corr(), mask=mask, annot=True, cmap="coolwarm")

plt.title("Correlation Heatmap")
plt.show()

# 8. Categorical Feature Analysis

In [ ]:
cat_cols = df.select_dtypes(include="object").columns

for col in cat_cols:
    plt.figure(figsize=(5,4))
    means = df.groupby(col)[TARGET_COL].mean().sort_values(ascending=False)
    means.plot(kind="bar")
    plt.title(f"Mean Score by {col}")
    plt.ylabel("Exam Score")
    plt.xticks(rotation=30)
    plt.show()

# 9. Scatter Plots

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.scatter(df["study_hours"], df[TARGET_COL], alpha=0.3)
plt.xlabel("Study Hours")
plt.ylabel("Exam Score")
plt.title("Study Hours vs Score")

plt.subplot(1,2,2)
plt.scatter(df["class_attendance"], df[TARGET_COL], alpha=0.3)
plt.xlabel("Attendance")
plt.ylabel("Exam Score")
plt.title("Attendance vs Score")

plt.show()


# 📌 Key Insights:

1. Study hours show strong positive correlation with exam score.
2. Attendance also positively impacts performance.
3. Some categorical features influence results moderately.
4. Data has minimal missing values and is relatively clean.

🎯 Conclusion:
EDA reveals important factors affecting student performance and prepares
the dataset for machine learning modeling.


# 10. Feature and Target Split

In [ ]:
TARGET_COL = "exam_score"  # make sure this matches your dataset

X = df.drop(TARGET_COL, axis=1)
y = df[TARGET_COL]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

# 11. Encoding Categorical Variable 

In [ ]:
from sklearn.preprocessing import LabelEncoder

cat_cols = X.select_dtypes(include="object").columns

le = LabelEncoder()
for col in cat_cols:
    X[col] = le.fit_transform(X[col])

X.head()

# 12. Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# 13. Model 1: Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

# 14. Model 2: Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=50,
    max_depth=10,
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

# 15. Evaluation Metrics

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

def evaluate(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return r2, mae, rmse

# 16. Model Comparison

In [ ]:
lr_results = evaluate(y_test, y_pred_lr)
rf_results = evaluate(y_test, y_pred_rf)

import pandas as pd

results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "R2 Score": [lr_results[0], rf_results[0]],
    "MAE": [lr_results[1], rf_results[1]],
    "RMSE": [lr_results[2], rf_results[2]]
})

results

# 17. Feature Importance

In [ ]:
import matplotlib.pyplot as plt

importance = rf.feature_importances_
features = X.columns

plt.figure(figsize=(8,5))
plt.barh(features, importance)
plt.title("Feature Importance (Random Forest)")
plt.show()


📊 Feature Importance Interpretation:

Top features such as study hours and attendance have the highest impact
on student performance, aligning with our EDA findings.

This validates that behavioral factors significantly influence exam scores.


# 18. Cross-Validation

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(rf, X, y, cv=5, scoring="r2")

print("CV Scores:", cv_scores)
print("Average CV Score:", cv_scores.mean())


# 📌 Machine Learning Insights:

1. Random Forest performed better than Linear Regression,
   indicating presence of non-linear relationships.

2. Key features influencing performance include:
   - Study hours
   - Attendance
   - Other behavioral factors

3. Model shows good predictive capability based on R² score.

🎯 Final Conclusion:
Machine learning models can effectively predict student performance
and help identify key drivers for academic success.
